# Training models in half genotypes to infer on others

## Loading the Database

In [1]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import numpy as np
import random

In [ ]:
IMG_SIZE = 512
BATCH_SIZE = 8
NUM_CLASSES = 1
CROP = "sorghum"  # "wheat" or "sorghum" or "corn"
#random.seed(1103)  # Set seed for reproducibility
all_genotypes = list(range(1, 81))
#awls_genotypes = [25, 26, 27, 29, 33, 34, 35, 36, 37, 38, 39]
broad_genotypes = [1, 3, 8, 11, 14, 15, 18, 20, 23, 25, 26, 27, 28, 29, 37, 39, 44, 46, 47, 49, 50, 51, 52, 55, 64, 65, 67, 68, 72, 76, 77, 79]
#train_genotypes = awls_genotypes
train_genotypes = [g for g in all_genotypes if g not in broad_genotypes]
print(f"🧬 Using train genotypes: {train_genotypes}")
GENOTYPE = "second_half"  # "first_half" or "second_half"

dataset_path = f"../data/{CROP}/"

class SegmentationDataset(Dataset):
    def __init__(self, root_dir, train=True, collection_dates=None, genotypes=None):
        self.root_dir = root_dir
        self.train = train
        self.img_dir = os.path.join(root_dir, "train/images" if train else "test/images")
        self.mask_dir = os.path.join(root_dir, "train/masks" if train else "test/masks")

        # List all image files
        image_files = sorted([
            f for f in os.listdir(self.img_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        mask_files = sorted([
            f for f in os.listdir(self.mask_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])

        # Filter by collection date if provided
        if collection_dates is not None:
            if isinstance(collection_dates, (str, int)):
                collection_dates = [str(collection_dates)]
            image_files = [f for f in image_files if f.split('_')[0] in collection_dates]
            mask_files = [f for f in mask_files if f.split('_')[0] in collection_dates]
        
        # Filter by genotype if provided
        if genotypes is not None:
            genotypes = set(str(g) for g in genotypes)
            image_files = [f for f in image_files if f.split('_')[1] in genotypes]
            mask_files = [f for f in mask_files if f.split('_')[1] in genotypes]
        print(image_files)
        assert len(image_files) == len(mask_files), \
            f"Image/Mask count mismatch: {len(image_files)} vs {len(mask_files)}"

        self.image_files = image_files
        self.mask_files = mask_files

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.image_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # grayscale mask

        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        mask = torch.from_numpy(np.array(mask)).float().unsqueeze(0)
        mask = (mask > 0).float()

        return image, mask

train_dataset = SegmentationDataset(dataset_path, train=True, genotypes=train_genotypes)
test_dataset = SegmentationDataset(dataset_path, train=False, genotypes=train_genotypes)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"✅ Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")

**Wheat**

first_half = [25, 26, 27, 29, 33, 34, 35, 36, 37, 38, 39] - Awls presence

second_half = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 28, 30, 31, 32, 40] - Not having awls

**Sorghum**

first_half = [1, 3, 8, 11, 14, 15, 18, 20, 23, 25, 26, 27, 28, 29, 37, 39, 44, 46, 47, 49, 50, 51, 52, 55, 64, 65, 67, 68, 72, 76, 77, 79] - Open Architecture

second_half = [2, 4, 5, 6, 7, 9, 10, 12, 13, 16, 17, 19, 21, 22, 24, 30, 31, 32, 33, 34, 35, 36, 38, 40, 41, 42, 43, 45, 48, 53, 54, 56, 57, 58, 59, 60, 61, 62, 63, 66, 69, 70, 71, 73, 74, 75, 78, 80] - Not open architecture

**Corn**

first_half = [103, 122, 120, 37, 50, 21, 43, 157, 101, 130, 106, 89, 39, 66, 30, 113, 35, 83, 117, 158, 134, 148, 15, 91, 111, 121, 24, 159, 3, 156, 93, 128, 8, 94, 108, 86, 26, 112, 110, 129, 84, 118, 78, 23, 95, 14, 53, 38, 42, 32, 136, 36, 34, 116, 25, 160, 52, 4, 140, 145, 58, 138, 33, 152, 63, 153, 61, 124, 27, 97, 72, 79, 146, 150, 7, 10, 81, 46, 9, 127]

second_half = [1, 2, 5, 6, 11, 12, 13, 16, 17, 18, 19, 20, 22, 28, 29, 31, 40, 41, 44, 45, 47, 48, 49, 51, 54, 55, 56, 57, 59, 60, 62, 64, 65, 67, 68, 69, 70, 71, 73, 74, 75, 76, 77, 80, 82, 85, 87, 88, 90, 92, 96, 98, 99, 100, 102, 104, 105, 107, 109, 114, 115, 119, 123, 125, 126, 131, 132, 133, 135, 137, 139, 141, 142, 143, 144, 147, 149, 151, 154, 155]

## Importing Models

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


### U-NET

In [8]:
import sys
sys.path.append('../')
from utils.models.uNet import UNet

channels = [32, 64, 128, 256, 512]
model = UNet(in_channels=3, out_channels=1, channels=channels, bilinear=True, use_batchnorm=True)
model.to(device)
modelName = "U-NET"

### SegFormer

In [6]:
import sys
sys.path.append('../')
from utils.models.SegFormer import segformer

model = segformer(in_channels = 3, num_classes = 1)
model.to(device)
modelName = "SegFormer"

## Training

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
from tqdm import tqdm

In [8]:
# --- Configuration ---
EPOCHS = 200
early_stopping_patience = 5
best_val_loss = float('inf') 
patience_counter = 0
scaler = torch.amp.GradScaler('cuda')

# --- Loss & Optimizer ---
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

checkpoint_path = os.path.join("../models/", CROP + "_" + modelName + "_" + GENOTYPE + "_seg.pt")
print(f"Model checkpoints will be saved to: {checkpoint_path}")

# --- Tracking ---
train_losses, val_losses = [], []

Model checkpoints will be saved to: ../models/sorghum_SegFormer_second_half_seg.pt


In [9]:
# --- Training Loop ---
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    with tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch") as tepoch:
        for inputs, labels in tepoch:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                    print("NaN or Inf in model outputs!")
                if torch.isnan(labels).any() or torch.isinf(labels).any():
                    print("NaN or Inf in labels!")
                if labels.sum() == 0:
                    print("Skipping batch with all empty masks")
                    continue
                loss = loss_fn(outputs, labels)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            tepoch.set_postfix(loss=running_loss / (len(tepoch)))
    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        with tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - Validation", unit="batch") as vepoch:
            for inputs, labels in vepoch:
                inputs, labels = inputs.to(device), labels.to(device)
                with torch.amp.autocast('cuda'):
                    outputs = model(inputs)
                    if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                        print("NaN or Inf in model outputs!")
                    if torch.isnan(labels).any() or torch.isinf(labels).any():
                        print("NaN or Inf in labels!")
                    if labels.sum() == 0:
                        print("Skipping batch with all empty masks")
                        continue
                    loss = loss_fn(outputs, labels)
                    preds = (torch.sigmoid(outputs) > 0.5).float()
                val_correct += (preds == labels).sum().item()
                val_total += labels.numel()
                val_acc = val_correct / val_total if val_total > 0 else 0
                val_loss += loss.item()
                vepoch.set_postfix(loss=val_loss / (len(vepoch)))

    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1} - Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.4f}%")

    # --- Scheduler step (use validation loss) ---
    lr_scheduler.step(avg_val_loss)

    # --- Checkpointing ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), checkpoint_path)
        print(f"✅ Model improved (Val Loss={avg_val_loss:.3f}), saved to {checkpoint_path}")
    else:
        patience_counter += 1
        print(f"🔴 No improvement, patience counter: {patience_counter}")

    if patience_counter >= early_stopping_patience:
        print("🛑 Early stopping triggered!")
        break

Epoch 1/200 - Validation: 100%|██████████| 471/471 [01:05<00:00,  7.17batch/s, loss=0.0461]


Epoch 1 - Val Loss: 0.0461 - Val Acc: 0.9825%
✅ Model improved (Val Loss=0.046), saved to ../models/sorghum_SegFormer_second_half_seg.pt


Epoch 2/200 - Validation: 100%|██████████| 471/471 [00:53<00:00,  8.85batch/s, loss=0.0342] 


Epoch 2 - Val Loss: 0.0342 - Val Acc: 0.9875%
✅ Model improved (Val Loss=0.034), saved to ../models/sorghum_SegFormer_second_half_seg.pt


Epoch 3/200 - Validation: 100%|██████████| 471/471 [00:50<00:00,  9.34batch/s, loss=0.0323] 


...

Epoch 16 - Val Loss: 0.0282 - Val Acc: 0.9907%
🔴 No improvement, patience counter: 3


Epoch 17/200 - Validation: 100%|██████████| 471/471 [00:52<00:00,  9.00batch/s, loss=0.0247] 


Epoch 17 - Val Loss: 0.0247 - Val Acc: 0.9909%
🔴 No improvement, patience counter: 4


Epoch 18/200 - Validation: 100%|██████████| 471/471 [00:51<00:00,  9.14batch/s, loss=0.0252] 

Epoch 18 - Val Loss: 0.0252 - Val Acc: 0.9913%
🔴 No improvement, patience counter: 5
🛑 Early stopping triggered!
